In [1]:
# Malayalam Speech Analysis and Transcription System - SIMPLE BATCH (analyze + store + next + clear)
# Google Colab friendly. Save as malayalam_speech_analysis.py and run in Colab.

# ==================== INSTALLATION ====================
# In Colab cell run:
# !pip install gradio transformers datasets librosa soundfile jiwer reportlab -q
# !pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

# ==================== IMPORTS ====================
import gradio as gr
import torch
import librosa
import numpy as np
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import os
import re
import unicodedata
from datetime import datetime
import io
from google.colab import drive

# ==================== MOUNT GOOGLE DRIVE ====================
print("🔗 Mounting Google Drive...")
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
print("✅ Google Drive mounted!")

# ==================== CONFIGURATION ====================
MODEL_PATH = '/content/drive/MyDrive/whisper_speech_error_detector_v2'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ==================== GROUND TRUTH PHONEME MAPPINGS ====================
WORD_MAPPING = {
    'Annann': {
        'malayalam': 'അണ്ണാൻ',
        'english': 'squirrel',
        'target_ipa': '/aɳɳa:n/',
        'correct_production': '/aɳɳa:n/',
        'target_sound': '/a/'
    },
    'AANAA': {
        'malayalam': 'ആന',
        'english': 'elephant',
        'target_ipa': '/a:na/',
        'correct_production': '/a:na/',
        'target_sound': '/a:/'
    },
    'Ellah': {
        'malayalam': 'ഇല',
        'english': 'leaf',
        'target_ipa': '/ila/',
        'correct_production': '/ila/',
        'target_sound': '/i/'
    },
    'EECHA': {
        'malayalam': 'ഈച്ച',
        'english': 'fly',
        'target_ipa': '/i:tʃ͡a/',
        'correct_production': '/i:tʃ͡a/',
        'target_sound': '/i:/'
    },
    'ulli': {
        'malayalam': 'ഉള്ളി',
        'english': 'onion',
        'target_ipa': '/uɭɭi/',
        'correct_production': '/uɭɭi/',
        'target_sound': '/u/'
    },
    'unjal': {
        'malayalam': 'ഊഞ്ഞാൽ',
        'english': 'swing',
        'target_ipa': '/u:ɲɲa:l/',
        'correct_production': '/u:ɲɲa:l/',
        'target_sound': '/u:/'
    },
    'eli': {
        'malayalam': 'എലി',
        'english': 'rat',
        'target_ipa': '/eli/',
        'correct_production': '/eli/',
        'target_sound': '/e/'
    },
    'eani': {
        'malayalam': 'ഏണി',
        'english': 'ladder',
        'target_ipa': '/e:ɳi/',
        'correct_production': '/e:ɳi/',
        'target_sound': '/e:/'
    },
    'Onnu': {
        'malayalam': 'ഒന്ന്',
        'english': 'one',
        'target_ipa': '/on̪n̪ə/',
        'correct_production': '/on̪n̪ə/',
        'target_sound': '/o/'
    },
    'Ola': {
        'malayalam': 'ഓല',
        'english': 'palm leaf',
        'target_ipa': '/o:la/',
        'correct_production': '/o:la/',
        'target_sound': '/o:/'
    },
    'Kuda': {
        'malayalam': 'കുട',
        'english': 'umbrella',
        'target_ipa': '/kuɖa/',
        'correct_production': '/kuɖa/',
        'target_sound': '/k/'
    },
    'Thakkol': {
        'malayalam': 'താക്കോല്‍',
        'english': 'key',
        'target_ipa': '/t̪a:kol/',
        'correct_production': '/t̪a:kol/',
        'target_sound': '/k/'
    },
    'Gandhiji': {
        'malayalam': 'ഗാന്ധിജി',
        'english': 'Gandhi',
        'target_ipa': '/ga:n̪d̪idʒ͡i/',
        'correct_production': '/ga:n̪d̪idʒ͡i/',
        'target_sound': '/ɡ/'
    },
    'Bag': {
        'malayalam': 'ബാഗ്',
        'english': 'bag',
        'target_ipa': '/ba:g/',
        'correct_production': '/ba:g/',
        'target_sound': '/ɡ/'
    },
    'Thatha': {
        'malayalam': 'തത്ത',
        'english': 'parrot',
        'target_ipa': '/t̪at̪a/',
        'correct_production': '/t̪at̪a/',
        'target_sound': '/ṯ/'
    },
    'Mothiram': {
        'malayalam': 'മോതിരം',
        'english': 'ring',
        'target_ipa': '/mo:t̪iɽam/',
        'correct_production': '/mo:t̪iɽam/',
        'target_sound': '/ṯ/'
    },
    'Nakshathram': {
        'malayalam': 'നക്ഷത്രം',
        'english': 'star',
        'target_ipa': '/n̪akʂat̪ram/',
        'correct_production': '/n̪akʂat̪ram/',
        'target_sound': '/n̪/'
    }
}

# Build Malayalam-to-key mapping for dropdown (Malayalam labels shown in UI)
MALAYALAM_TO_KEY = {v['malayalam']: k for k, v in WORD_MAPPING.items()}
MALAYALAM_CHOICES = list(MALAYALAM_TO_KEY.keys())

# ==================== MODEL LOADING ====================
def load_model():
    """Load the fine-tuned Whisper model from Google Drive"""
    try:
        print(f"🤖 Loading model from: {MODEL_PATH}")
        processor = WhisperProcessor.from_pretrained(MODEL_PATH)
        model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH)
        model.to(DEVICE)
        model.eval()
        print("✅ Model loaded successfully!")
        return processor, model
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("⚠️ Attempting to load base Whisper model as fallback...")
        try:
            processor = WhisperProcessor.from_pretrained("openai/whisper-small")
            model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
            model.to(DEVICE)
            model.eval()
            print("✅ Base model loaded as fallback")
            return processor, model
        except Exception as e2:
            print(f"❌ Failed to load any model: {e2}")
            return None, None

processor, model = load_model()

# ==================== PHONEME ANALYSIS FUNCTIONS (NEW S-O-A-D ENGINE) ====================

# -------------------------------
# 1️⃣ IPA TOKENIZER  (regex-based, handles all Malayalam IPA clusters)
# -------------------------------

# COMBINING DIACRITIC codepoints that must stay attached to the preceding base char
_COMBINING = '\u0320-\u036F\u1DC0-\u1DFF\u20D0-\u20FF'   # broad combining ranges
# U+0331 = combining minus below (used for dental ̪)

# Ordered list of multi-character tokens — longest match first
_MULTI_TOKENS = [
    # affricates (tie-bar variants)
    'tʃ͡', 'dʒ͡', 't͡ʃ', 'd͡ʒ',
    # dental + combining-minus-below
    'n̪', 't̪', 'd̪', 's̪', 'l̪', 'r̪',
    # long vowels
    'a:', 'i:', 'u:', 'e:', 'o:', 'ə:',
]

def tokenize_ipa(ipa_string):
    """
    Robustly tokenize an IPA string into a list of phoneme tokens.
    Strips surrounding slashes, then greedily matches longest tokens first.
    Each token is a single phoneme (possibly multi-codepoint).
    """
    s = ipa_string.strip().strip('/')
    tokens = []
    i = 0
    while i < len(s):
        # Skip spaces
        if s[i] == ' ':
            i += 1
            continue
        matched = False
        # Try multi-char tokens (longest first)
        for mt in _MULTI_TOKENS:
            if s[i:i+len(mt)] == mt:
                tokens.append(mt)
                i += len(mt)
                matched = True
                break
        if not matched:
            # Take one base character plus any immediately following combining diacritics
            ch = s[i]
            i += 1
            while i < len(s) and unicodedata.combining(s[i]):
                ch += s[i]
                i += 1
            tokens.append(ch)
    return tokens

# -------------------------------
# 2️⃣ LEVENSHTEIN MATRIX
# -------------------------------
def phoneme_levenshtein(target_tokens, prod_tokens):
    m = len(target_tokens)
    n = len(prod_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if target_tokens[i - 1] == prod_tokens[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,           # omission (target phoneme deleted)
                dp[i][j - 1] + 1,           # addition (extra phoneme inserted)
                dp[i - 1][j - 1] + cost     # match or substitution
            )
    return dp

# -------------------------------
# 3️⃣ FEATURE GROUPS FOR DISTORTION DETECTION
# -------------------------------
retroflex_group = {'ɳ', 'ɖ', 'ɭ', 'ɽ', 'ʈ'}
dental_group    = {'n̪', 'd̪', 't̪', 'l̪', 's̪', 'r̪'}
nasal_group     = {'m', 'n', 'ɳ', 'ɲ', 'n̪', 'ŋ'}
voicing_pairs   = {('p','b'), ('t','d'), ('k','g'), ('t̪','d̪'),
                   ('ʈ','ɖ'), ('tʃ͡','dʒ͡'), ('t͡ʃ','d͡ʒ')}

def classify_substitution(target, produced):
    """
    Determine whether a mismatch is a true Substitution or a Distortion.
    Distortion = acoustically similar, wrong in one phonetic feature only.
    """
    # 1. Short ↔ long vowel (length change only)
    if target.replace(':', '') == produced.replace(':', '') and target != produced:
        return "Distortion (vowel length)"

    # 2. Retroflex ↔ dental (place of articulation)
    if (target in retroflex_group and produced in dental_group) or \
       (produced in retroflex_group and target in dental_group):
        return "Distortion (retroflex↔dental)"

    # 3. Voicing difference only
    if (target, produced) in voicing_pairs or (produced, target) in voicing_pairs:
        return "Distortion (voicing)"

    # 4. Within nasals (place change only)
    if target in nasal_group and produced in nasal_group:
        return "Distortion (nasal place)"

    return "Substitution"

# -------------------------------
# 4️⃣ BACKTRACK TO EXTRACT S-O-A-D
# -------------------------------
def extract_operations(dp, target_tokens, prod_tokens):
    """
    Backtrack through the Levenshtein DP matrix to label each edit operation.
    Priority order at each cell: Match > Substitution/Distortion > Omission > Addition
    This prevents mis-labelling length-mark omissions as substitutions.
    """
    i = len(target_tokens)
    j = len(prod_tokens)
    substitutions = []
    omissions     = []
    additions     = []
    distortions   = []

    while i > 0 or j > 0:
        if i > 0 and j > 0 and target_tokens[i - 1] == prod_tokens[j - 1]:
            # Exact match — move diagonally, no error
            i -= 1
            j -= 1

        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
            # Substitution or Distortion
            t = target_tokens[i - 1]
            p = prod_tokens[j - 1]
            result = classify_substitution(t, p)
            if "Distortion" in result:
                distortions.append((t, p, result))
            else:
                substitutions.append((t, p))
            i -= 1
            j -= 1

        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            # Omission — target phoneme was not produced
            omissions.append(target_tokens[i - 1])
            i -= 1

        elif j > 0 and dp[i][j] == dp[i][j - 1] + 1:
            # Addition — extra phoneme produced
            additions.append(prod_tokens[j - 1])
            j -= 1

        else:
            # Fallback (should not normally occur)
            if i > 0:
                i -= 1
            else:
                j -= 1

    return substitutions[::-1], omissions[::-1], additions[::-1], distortions[::-1]

# -------------------------------
# 5️⃣ MASTER ANALYSIS FUNCTION
# -------------------------------
def analyze_S_O_A_D(target_word, production):
    """
    Compare target IPA with produced IPA and return categorised errors.
    Returns a dict with keys: Substitution, Omission, Addition, Distortion.
    Also returns token lists for debugging.
    """
    target_tokens = tokenize_ipa(target_word)
    prod_tokens   = tokenize_ipa(production)

    # Debug printout (visible in Colab logs)
    print(f"  [SOAD] Target tokens : {target_tokens}")
    print(f"  [SOAD] Produced tokens: {prod_tokens}")

    dp = phoneme_levenshtein(target_tokens, prod_tokens)
    S, O, A, D = extract_operations(dp, target_tokens, prod_tokens)
    return {
        "Substitution": S,
        "Omission":     O,
        "Addition":     A,
        "Distortion":   D
    }

# ==================== TRANSCRIPTION ====================
def transcribe_audio(audio_path):
    """Transcribe audio using the fine-tuned model"""
    if model is None or processor is None:
        return "❌ Model not loaded"
    try:
        audio, sr = librosa.load(audio_path, sr=16000)
        input_features = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt"
        ).input_features.to(DEVICE)

        with torch.no_grad():
            predicted_ids = model.generate(input_features)

        transcription = processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True
        )[0].strip()

        return transcription

    except Exception as e:
        return f"❌ Error: {str(e)}"

# ==================== MAIN ANALYSIS ====================
def analyze_single_record(selected_mal_word, audio_path):
    """
    Analyze one audio for the selected Malayalam word.
    Returns:
      result_table_md, error_type_md, error_description_md, error_details_text, analysis_record (dict)
    """
    if not selected_mal_word:
        return ("⚠️ Please select a word first!", "", "", "", None)
    if audio_path is None:
        return ("⚠️ Please record or upload audio!", "", "", "", None)

    key = MALAYALAM_TO_KEY[selected_mal_word]
    word_info = WORD_MAPPING[key]
    target_ipa = word_info['target_ipa']
    correct_production = word_info['correct_production']
    target_sound = word_info['target_sound']

    transcription = transcribe_audio(audio_path)
    if transcription.startswith("❌"):
        return (transcription, "", "", "", None)

    # ── NEW: run S-O-A-D analysis ──────────────────────────────────────────────
    soad = analyze_S_O_A_D(correct_production, transcription)

    # Build a compact error-code list (only non-empty categories)
    code_map = {'Substitution': 'S', 'Omission': 'O', 'Addition': 'A', 'Distortion': 'D'}
    error_codes = [code_map[k] for k in ('Substitution', 'Omission', 'Addition', 'Distortion')
                   if soad[k]]


    # Build human-readable detail lines
    error_details_lines = []
    if soad['Substitution']:
        for t, p in soad['Substitution']:
            error_details_lines.append(f"• [S] Substitution: '{t}' → '{p}'")
    if soad['Omission']:
        for ph in soad['Omission']:
            error_details_lines.append(f"• [O] Omission: '{ph}' was omitted")
    if soad['Addition']:
        for ph in soad['Addition']:
            error_details_lines.append(f"• [A] Addition: extra '{ph}' inserted")
    if soad['Distortion']:
        for t, p, reason in soad['Distortion']:
            error_details_lines.append(f"• [D] Distortion ({reason}): '{t}' → '{p}'")
    if not error_details_lines:
        error_details_lines.append("✅ No phoneme-level errors detected.")
    # ──────────────────────────────────────────────────────────────────────────

    # Prepare result display (markdown)
    result_table = f"""
## 📋 Analysis Results

| Field | Value |
|---|---|
| Selected Word | {selected_mal_word} |
| English | {word_info['english']} |
| Target Phoneme | `{target_ipa}` |
| Target Sound | `{target_sound}` |
| Expected Production | `{correct_production}` |
| Actual Production | `{transcription}` |
| Error Type(s) | {', '.join(error_codes) if error_codes else 'None'} |
    """

    if error_codes:
        error_type_display = f"## ❌ Error Detected: {', '.join(error_codes)}"
    else:
        error_type_display = "## ✅ No Errors - Perfect Production!"

    desc_map = {
        'S': '**Substitution**: Replacing one sound with another.',
        'O': '**Omission**: Leaving out a sound.',
        'D': '**Distortion**: Producing a sound inaccurately but recognizably.',
        'A': '**Addition**: Adding an unnecessary sound.'
    }
    if error_codes:
        error_description = "  ".join([desc_map[c] for c in error_codes])
    else:
        error_description = "✅ No errors detected."

    error_list_text = "\n".join(error_details_lines)

    analysis_data = {
        'word_key': key,
        'word_malayalam': selected_mal_word,
        'english': word_info['english'],
        'target_ipa': target_ipa,
        'target_sound': target_sound,
        'expected': correct_production,
        'produced': transcription,
        'error_codes': error_codes,
        'soad': soad,
        'error_details': error_details_lines,
        'audio_path': audio_path,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    return (result_table, error_type_display, error_description, error_list_text, analysis_data)

# ==================== BATCH FORMATTING ====================
def format_batch_markdown(batch):
    """Create a markdown table summarizing the current batch"""
    if not batch:
        return "ℹ️ Batch is empty."
    md = "| SL No | Target Sound | Target Word | Production | Errors |\n"
    md += "|---:|---|---|---|---|\n"
    for i, rec in enumerate(batch, start=1):
        errs = ",".join(rec.get('error_codes', [])) if rec.get('error_codes') else "-"
        produced = rec.get('produced', '')
        produced_short = (produced[:40] + '...') if len(produced) > 43 else produced
        md += f"| {i} | `{rec.get('target_sound','')}` | {rec.get('word_malayalam','')} | `{produced_short}` | {errs} |\n"
    return md

# ==================== GRADIO INTERFACE (SIMPLIFIED) ====================
def create_interface():
    with gr.Blocks(
        theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="blue"),
        title="Speech Analysis and Transcription System"
    ) as demo:
        gr.HTML("""
            <div style="text-align: center; padding: 16px;">
                <h1 style="color: #1e40af; margin-bottom: 4px;">🎙️ Speech Analysis and Transcription System</h1>
                <p style="color: #6b7280; font-size: 14px;">Analyze → Store into batch → Next</p>
            </div>
        """)

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 📝 Step 1: Select Target Word (Malayalam labels)")

                word_dropdown = gr.Dropdown(
                    choices=MALAYALAM_CHOICES,
                    label="Target Word (Malayalam)",
                    info="Select the Malayalam word to be assessed"
                )

                word_info_display = gr.Markdown("ℹ️ Please select a word to see details")

                def update_word_info(mal_word):
                    if mal_word and mal_word in MALAYALAM_TO_KEY:
                        key = MALAYALAM_TO_KEY[mal_word]
                        info = WORD_MAPPING[key]
                        return f"""
**Selected Word:** {mal_word}
**English Translation:** {info['english']}
**Target Phoneme (IPA):** `{info['target_ipa']}`
**Target Sound:** `{info['target_sound']}`
**Expected Production:** `{info['correct_production']}`
                        """
                    return "ℹ️ Please select a word to see details"

                word_dropdown.change(fn=update_word_info, inputs=[word_dropdown], outputs=[word_info_display])

                gr.Markdown("### 🎤 Step 2: Record or Upload Audio")
                audio_input = gr.Audio(
                    sources=["microphone", "upload"],
                    type="filepath",
                    label="Audio Input (Supports: MP3, WAV, M4A, FLAC, OGG, etc.)",
                    format="wav"
                )

                with gr.Row():
                    analyze_btn     = gr.Button("🔍 Analyze", variant="primary")
                    next_word_btn   = gr.Button("➕ Next Word (Store & Clear)", variant="primary")
                    clear_batch_btn = gr.Button("🔄 Clear Batch", variant="secondary")

            with gr.Column(scale=1):
                gr.Markdown("### 📊 Analysis Results (Current)")
                result_table       = gr.Markdown("ℹ️ Results will appear here after analysis")
                error_type_display = gr.Markdown("")
                error_description  = gr.Markdown("")
                gr.Markdown("#### 📋 Detailed Error Analysis")
                error_details_list = gr.Textbox(
                    label="Error Details",
                    lines=6,
                    interactive=False,
                    placeholder="Detailed phoneme-level error information will appear here..."
                )

                gr.Markdown("### 📚 Batch (Stored Results)")
                batch_display = gr.Markdown("ℹ️ No stored results yet.")

        # States
        last_analysis = gr.State(None)
        batch_state   = gr.State([])

        # Analyze button
        analyze_btn.click(
            fn=analyze_single_record,
            inputs=[word_dropdown, audio_input],
            outputs=[result_table, error_type_display, error_description, error_details_list, last_analysis]
        )

        # Next Word: add last_analysis to batch, clear UI
        def next_word_action(last_analysis_data, current_batch):
            if not last_analysis_data:
                return ("⚠️ No analysis to add. Please analyze first.", "", "", "", None,
                        current_batch, format_batch_markdown(current_batch), None, None)
            new_batch = list(current_batch) if current_batch else []
            new_batch.append(last_analysis_data)
            batch_md = format_batch_markdown(new_batch)
            return ("ℹ️ Results will appear here after analysis", "", "", "", None,
                    new_batch, batch_md, None, None)

        next_word_btn.click(
            fn=next_word_action,
            inputs=[last_analysis, batch_state],
            outputs=[result_table, error_type_display, error_description, error_details_list,
                     last_analysis, batch_state, batch_display, word_dropdown, audio_input]
        )

        # Clear Batch button
        def clear_batch_action():
            return [], "ℹ️ Batch cleared."

        clear_batch_btn.click(
            fn=clear_batch_action,
            inputs=[],
            outputs=[batch_state, batch_display]
        )

        gr.Markdown("""
---
### 📋 How to Use (Simple Batch):
1. Select a Malayalam word.
2. Record or upload audio for that word.
3. Click **Analyze** to run analysis. Results appear as phoneme-level S/O/A/D feedback.
4. Click **Next Word (Store & Clear)** to append the result to the batch and clear the UI for the next word.
5. Use **Clear Batch** to remove all stored results.

**Error codes:** S = Substitution · O = Omission · A = Addition · D = Distortion

---
*For Clinical Use Only*
        """)
    return demo

# ==================== MAIN EXECUTION ====================
if __name__ == "__main__":
    print("\n" + "=" * 80)
    print("MALAYALAM SPEECH ANALYSIS SYSTEM - STARTING (SIMPLE BATCH MODE)")
    print("=" * 80)

    if model is not None and processor is not None:
        print("✅ All systems ready!")
        print(f"📊 Loaded {len(WORD_MAPPING)} target words")
        print(f"🎯 Device: {DEVICE}")
        demo = create_interface()
        demo.launch(share=True, debug=True)
    else:
        print("❌ Failed to initialize system. Please check model path.")
        print(f"Expected model location: {MODEL_PATH}")

🔗 Mounting Google Drive...
Mounted at /content/drive
✅ Google Drive mounted!
Using device: cuda
🤖 Loading model from: /content/drive/MyDrive/whisper_speech_error_detector_v2


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

✅ Model loaded successfully!

MALAYALAM SPEECH ANALYSIS SYSTEM - STARTING (SIMPLE BATCH MODE)
✅ All systems ready!
📊 Loaded 17 target words
🎯 Device: cuda


/tmp/ipykernel_5182/2392245281.py:512: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2c5d6f5499d1a1c93b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2c5d6f5499d1a1c93b.gradio.live
